<a href="https://colab.research.google.com/github/KapoorAkshit18/PGCP_VLSI/blob/main/AI_Sessions/Session_03/AIs02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
gemini_key=userdata.get('ai_cdac_01')

In [6]:
pip -q install langchain langchain-google-genai langchain-chroma langchain-community

In [3]:
pip -q install opentelemetry-api==1.42.1 opentelemetry-sdk==1.42.1 chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# Initialize Google Generative AI Embeddings with your API key and model
embeddings = GoogleGenerativeAIEmbeddings(model="text-embedding-004", google_api_key=gemini_key)

Next, let's create some dummy documents and load them into Chroma. For a real-world scenario, you would load your actual data here.

In [7]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# Create dummy documents
dummy_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is rapidly advancing.",
    "LangChain provides tools for building LLM applications.",
    "Chroma is a popular open-source vector database."
]

# Split texts into documents (simple example, can use TextLoader/TextSplitter for more complex data)
documents = [Document(page_content=text) for text in dummy_texts]

# Re-initialize Google Generative AI Embeddings within this cell
# using the recommended model for embedding
# The previous model 'text-embedding-004' was not found or supported for embedding.
embeddings_fixed = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=gemini_key)


# Initialize Chroma with the documents and embeddings
# We'll create an in-memory Chroma instance for this demonstration.
vectordb = Chroma.from_documents(
    documents,
    embeddings_fixed, # Use the fixed embeddings object
    collection_name="my_documents_collection"
)

print("Chroma vector store initialized with documents.")

Chroma vector store initialized with documents.


In [8]:
import google.generativeai as genai

# Configure with your API key
genai.configure(api_key=gemini_key)

print("Available Generative AI Models:")
for m in genai.list_models():
    # Check if the model supports embedding and print its name
    if 'embedContent' in m.supported_generation_methods:
        print(f"  - {m.name} (Supports embedding)")


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available Generative AI Models:
  - models/gemini-embedding-001 (Supports embedding)
  - models/gemini-embedding-2-preview (Supports embedding)
  - models/gemini-embedding-2 (Supports embedding)


Now you can perform similarity searches against your Chroma vector store.

In [9]:
# Example: Perform a similarity search
query = "What is AI?"
results = vectordb.similarity_search(query, k=2)

print(f"\nSimilarity search results for query: '{query}'")
for i, doc in enumerate(results):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)


Similarity search results for query: 'What is AI?'

Document 1:
Artificial intelligence is rapidly advancing.

Document 2:
LangChain provides tools for building LLM applications.


In [45]:
pip install -U langchain langchain-core langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.15
    Uninstalling langchain-1.3.15:
      Successfully uninstalled langchain-1.3.15


RETRIVAL

In [11]:
# from langchain.chains import RetrievalQA
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_core.prompts import PromptTemplate

from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [43]:
# from langchain.chains import RetrievalQA
# from langchain_google_genai import GoogleGenerativeAI
# from langchain.prompts import PromptTemplate

# The 'vectordb' with dummy documents was created in cell 7951711d.
# We will use that existing vectordb to create the retriever.

# Retriever: top k = 5
retriever = vectordb.as_retriever(search_kwargs={"k": 5})

def retrieve_docs(query: str):
  return retriever.invoke(query)

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

print("Retriever initialized from existing vectordb.")
# Test the retriever (optional, but good for debugging)
# You can uncomment the line below to see what the retriever returns for a query.
# print(retrieve_docs("What is AI?"))

Retriever initialized from existing vectordb.


Next, we'll define a prompt template that guides the Large Language Model (LLM) on how to answer questions based on the retrieved context.

In [19]:
template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.
{context}
Question: {question}
Helpful Answer:"""

QA_CHAIN_PROMPT = ChatPromptTemplate.from_template(template)

Now, let's initialize the Google Generative AI model and create the `RetrievalQA` chain. This chain will take a question, retrieve relevant documents from the `vectordb`, and then use the LLM to generate an answer based on the retrieved information.

In [39]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter # Import itemgetter for more explicit key extraction

# Initialize the Google Generative AI LLM
llm = GoogleGenerativeAI(model="models/gemini-3.6-flash", google_api_key=gemini_key) # Updated model name to an available one

# Define the format_docs function (already present in cell lCGuGfAHV6VU, but for clarity)
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

# Create an LCEL-based Retrieval-Augmented Generation (RAG) chain
# Ensure 'retriever' from cell lCGuGfAHV6VU is used, assuming it's correctly defined and initialized.
qa_chain = (
    RunnableParallel(
        context=itemgetter("question") | retriever | format_docs, # Use itemgetter to pass only the question string to the retriever
        question=itemgetter("question") # Also use itemgetter to pass only the question string to the prompt
    )
    | QA_CHAIN_PROMPT
    | llm
    | StrOutputParser()
)

print("LCEL-based QA chain created successfully.")

LCEL-based QA chain created successfully.


Finally, let's test the `RetrievalQA` chain with a question related to our dummy documents.

In [40]:
question = "What is Chroma?"
# Call the LCEL chain using .invoke() and pass the question with the correct key.
answer = qa_chain.invoke({"question": question})

print(f"\nQuestion: {question}")
# The chain directly returns the answer as a string, not a dictionary.
print(f"Answer: {answer}")
# Source documents are not returned by the current LCEL chain configuration.
# If you need source documents, the qa_chain definition in the previous cell would need to be modified.
# print("\nSource Documents:")
# for doc in result['source_documents']:
#     print(f"- {doc.page_content}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Question: What is Chroma?
Answer: I don't know, as no context was provided to answer the question.


In [36]:
import google.generativeai as genai

genai.configure(api_key=gemini_key)

print("Available Generative Models (supporting generateContent):")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"  - {m.name}")

Available Generative Models (supporting generateContent):
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3.1-flash-lite
  - models/gemini-3-pro-image-preview
  - models/gemini-3-pro-image
  - models/nano-banana-pro-preview
  - models/gemini-3.1-flash-image-preview
  - models/gemini-3.1-flash-image
  - models/gemini-3.1-flash-lite-image
  - models/gemini-3.5-flash
  - models/gemini-3.5-flash-lite
  - models/gemini-omni-flash-preview
  - models/gemini-3.6-flash
  - models/gemini-3.7-flash
  - models/lyria-3-clip-preview

In [44]:
question = "Tell me about Artificial intelligence?"
# Call the LCEL chain using .invoke() and pass the question with the correct key.
answer = qa_chain.invoke({"question": question})

print(f"\nQuestion: {question}")
print(f"Answer: {answer}")
# The current LCEL chain (qa_chain) directly returns a string answer.
# It does not return source documents in its output.
# If you need source documents, the qa_chain definition would need to be modified to include them.
# For example, using RunnablePassthrough.assign with retrieved_docs.
# print("\nSource Documents:")
# for doc in result['source_documents']:
#     print(f"- {doc.page_content}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Question: Tell me about Artificial intelligence?
Answer: Since no context was provided with your prompt, I cannot answer based on specific text. However, generally speaking:

**Artificial Intelligence (AI)** refers to the simulation of human intelligence in computers and machines. It involves creating systems capable of performing tasks that typically require human cognition, such as:

* **Learning:** Acquiring information and rules for using it (machine learning, deep learning).
* **Reasoning:** Using rules to reach approximate or definite conclusions.
* **Problem-Solving & Decision-Making:** Analyzing data to make choices or solve complex problems.
* **Perception:** Understanding inputs like speech, visual scenes, or natural language (e.g., Computer Vision, Natural Language Processing).

Common applications of AI include virtual assistants (like Siri or Alexa), recommendation systems (like Netflix or Spotify), autonomous vehicles, generative AI tools (like ChatGPT), and medical diag